In [ ]:
import numpy as np
import tritonclient.http as httpclient
import matplotlib.pyplot as plt
from IPython.display import Audio, display

TRITON_URL = "localhost:8000"
MODEL_NAME = "qwen3_tts"
SAMPLE_RATE = 24000

In [ ]:
def synthesize(text: str, language: str = "english") -> np.ndarray:
    client = httpclient.InferenceServerClient(url=TRITON_URL)

    text_input = httpclient.InferInput("text", [1], "BYTES")
    text_input.set_data_from_numpy(np.array([text], dtype=object))

    lang_input = httpclient.InferInput("language", [1], "BYTES")
    lang_input.set_data_from_numpy(np.array([language], dtype=object))

    result = client.infer(
        model_name=MODEL_NAME,
        inputs=[text_input, lang_input],
        outputs=[httpclient.InferRequestedOutput("audio")],
    )
    return result.as_numpy("audio")

In [ ]:
audio = synthesize(
    text="oh, that one was fast. hey yeah",
    language="english",
)
print(f"Got {len(audio)} samples — {len(audio) / SAMPLE_RATE:.2f}s @ {SAMPLE_RATE} Hz")

In [ ]:
t = np.arange(len(audio)) / SAMPLE_RATE

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, audio, linewidth=0.3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Qwen3-TTS Waveform")
fig.tight_layout()
plt.show()

display(Audio(audio, rate=SAMPLE_RATE))